# NemoClaw Hermes on Brev

This notebook sets up a NemoClaw sandbox that runs Hermes Agent.

The default path uses NVIDIA-hosted inference through build.nvidia.com. This CPU Brev instance runs Docker, OpenShell, NemoClaw, and the Hermes sandbox.

Run the cells in order. The first build can take several minutes while Docker images are pulled and the Hermes sandbox image is prepared.

## Configuration

The NemoClaw Hermes reference architecture accepts a number of configuration options. Default values can be set through environment variables or running `nemoclaw onboard` will trigger an interactive wizard to guide you through selections.

```bash
export NEMOCLAW_AGENT=hermes
export NEMOCLAW_NON_INTERACTIVE=1
export NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE=1
export NEMOCLAW_SANDBOX_NAME=brev-hermes
export NVIDIA_API_KEY=<your build.nvidia.com key>
curl -fsSL https://www.nvidia.com/nemoclaw.sh | bash
```

Important options surfaced below:

| Option | Default | Meaning |
|---|---:|---|
| `NVIDIA_API_KEY` | prompt | NVIDIA Build API key from https://build.nvidia.com/settings/api-keys. |
| `NEMOCLAW_SANDBOX_NAME` | `brev-hermes` | Name of the OpenShell sandbox NemoClaw creates. |
| `NEMOCLAW_MODEL` | `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` | NVIDIA-hosted model used by Hermes. The smaller default keeps the first Brev onboarding validation fast; switch to `nvidia/nemotron-3-super-120b-a12b` after onboarding if you want the larger model. |
| `NEMOCLAW_PROVIDER` | `build` | Selects NVIDIA Endpoints during onboarding. |
| `NEMOCLAW_POLICY_TIER` | `balanced` | NemoClaw policy tier. Use `restricted` for tighter defaults or `permissive` while debugging. |
| `NEMOCLAW_SANDBOX_GPU` | `0` | Forces CPU-only sandbox creation. Keep this for Brev CPU instances. |
| `NEMOCLAW_DASHBOARD_BIND` | `0.0.0.0` | Lets the forwarded Hermes API port bind beyond loopback on remote hosts such as Brev. |
| `NEMOCLAW_SANDBOX_READY_TIMEOUT` | `900` | Wait budget in seconds for the sandbox to become ready. Increase on slow first builds. |
| `NEMOCLAW_ONBOARD_EXTRA_ARGS` | unset | Optional extra arguments for `nemohermes onboard`. Set to `--fresh` once if you are recovering from an interrupted or failed onboarding attempt. |

For a non-NVIDIA OpenAI-compatible endpoint, use `NEMOCLAW_PROVIDER=custom`, set `NEMOCLAW_ENDPOINT_URL`, and provide `COMPATIBLE_API_KEY` instead of the NVIDIA key. 

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import shlex

# Edit these defaults before running the cell if you want a different setup.
SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "brev-hermes")
# Keep first onboarding responsive on small Brev CPU instances.
MODEL = os.environ.get("NEMOCLAW_MODEL", "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning")
PROVIDER = os.environ.get("NEMOCLAW_PROVIDER", "build")
POLICY_TIER = os.environ.get("NEMOCLAW_POLICY_TIER", "balanced")
SANDBOX_GPU = os.environ.get("NEMOCLAW_SANDBOX_GPU", "0")
READY_TIMEOUT = os.environ.get("NEMOCLAW_SANDBOX_READY_TIMEOUT", "900")
DASHBOARD_BIND = os.environ.get("NEMOCLAW_DASHBOARD_BIND", "0.0.0.0")
RESOURCE_PROFILE = os.environ.get("NEMOCLAW_RESOURCE_PROFILE", "default")
ONBOARD_EXTRA_ARGS = os.environ.get("NEMOCLAW_ONBOARD_EXTRA_ARGS", "")

nvidia_api_key = os.environ.get("NVIDIA_API_KEY")
if not nvidia_api_key and PROVIDER == "build":
    nvidia_api_key = getpass("Paste your NVIDIA Build API key: ")

env = {
    "NEMOCLAW_AGENT": "hermes",
    "NEMOCLAW_PROVIDER": PROVIDER,
    "NEMOCLAW_NON_INTERACTIVE": "1",
    "NEMOCLAW_ACCEPT_THIRD_PARTY_SOFTWARE": "1",
    "NEMOCLAW_YES": "1",
    "NEMOCLAW_NO_EXPRESS": "1",
    "NEMOCLAW_SANDBOX_NAME": SANDBOX_NAME,
    "NEMOCLAW_MODEL": MODEL,
    "NEMOCLAW_POLICY_TIER": POLICY_TIER,
    "NEMOCLAW_SANDBOX_GPU": SANDBOX_GPU,
    "NEMOCLAW_SANDBOX_READY_TIMEOUT": READY_TIMEOUT,
    "NEMOCLAW_DASHBOARD_BIND": DASHBOARD_BIND,
    "NEMOCLAW_RESOURCE_PROFILE": RESOURCE_PROFILE,
}
if nvidia_api_key:
    env["NVIDIA_API_KEY"] = nvidia_api_key
if ONBOARD_EXTRA_ARGS:
    env["NEMOCLAW_ONBOARD_EXTRA_ARGS"] = ONBOARD_EXTRA_ARGS

# Optional custom endpoint support. Leave unset for the NVIDIA Build path.
for optional_key in ("NEMOCLAW_ENDPOINT_URL", "COMPATIBLE_API_KEY", "OPENAI_API_KEY"):
    if os.environ.get(optional_key):
        env[optional_key] = os.environ[optional_key]

env_path = Path("/tmp/nemoclaw-hermes-launchable.env")
env_path.write_text("\n".join(f"export {k}={shlex.quote(v)}" for k, v in env.items()) + "\n")
env_path.chmod(0o600)

redacted = {k: ("<set>" if "KEY" in k or "TOKEN" in k else v) for k, v in env.items()}
print(f"Wrote shell environment to {env_path}")
print("Effective setup:")
for key in sorted(redacted):
    print(f"  {key}={redacted[key]}")

## Prepare the Brev Host

NemoClaw needs Docker and a Linux host. Brev VM-mode instances normally include Docker, but this cell installs and starts it if needed. It also makes the Docker socket usable from the current Jupyter user so the same notebook session can continue without logging out and back in.

In [ ]:
%%bash
set -euo pipefail
source /tmp/nemoclaw-hermes-launchable.env

GATEWAY_PORT="${NEMOCLAW_GATEWAY_PORT:-8080}"

echo "Host: $(uname -a)"
echo "User: $(id)"

if ! command -v docker >/dev/null 2>&1; then
  echo "Installing Docker..."
  sudo apt-get update -qq
  sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq docker.io
fi

if command -v systemctl >/dev/null 2>&1; then
  sudo systemctl enable --now docker || true
else
  sudo service docker start || true
fi

sudo usermod -aG docker "$USER" 2>/dev/null || true
sudo chmod 666 /var/run/docker.sock 2>/dev/null || true

# Small Brev CPU instances can build the sandbox slowly and may run close to
# memory limits. Add swap once so Docker image builds have breathing room.
mem_mb=$(awk '/MemTotal:/ {print int($2/1024)}' /proc/meminfo)
swap_mb=$(awk '/SwapTotal:/ {print int($2/1024)}' /proc/meminfo)
if [ "$mem_mb" -lt 12000 ] && [ "$swap_mb" -eq 0 ] && [ ! -f /swapfile ]; then
  echo "Creating 8 GiB swapfile for sandbox image builds..."
  sudo fallocate -l 8G /swapfile || sudo dd if=/dev/zero of=/swapfile bs=1M count=8192 status=progress
  sudo chmod 600 /swapfile
  sudo mkswap /swapfile
  sudo swapon /swapfile
fi

# Brev images often run UFW. NemoClaw's OpenShell Docker-driver gateway
# listens on the host-side Docker bridge address (for example 172.18.0.1:8080),
# and UFW can block sandbox containers from reaching it. Allow only Docker
# bridge subnets to the configured gateway port.
if command -v ufw >/dev/null 2>&1 && sudo ufw status | grep -qi '^Status: active'; then
  echo "Allowing Docker bridge traffic to OpenShell gateway port ${GATEWAY_PORT}..."
  mapfile -t docker_subnets < <(
    docker network inspect $(docker network ls -q)       --format '{{range .IPAM.Config}}{{if .Subnet}}{{.Subnet}}{{"\n"}}{{end}}{{end}}' 2>/dev/null       | sed '/^$/d' | sort -u
  )
  # openshell-docker may not exist until onboarding starts, so include its
  # common default subnet as a fallback.
  docker_subnets+=("172.18.0.0/16")
  printf '%s
' "${docker_subnets[@]}" | sed '/^$/d' | sort -u | while read -r subnet; do
    sudo ufw allow from "$subnet" to any port "$GATEWAY_PORT" proto tcp || true
  done
fi

docker version --format 'Docker {{.Server.Version}}' || docker version
docker info --format 'Docker root: {{.DockerRootDir}}'
free -h
sudo ufw status || true


## Install NemoClaw and Run Hermes Onboarding

This is the main setup cell. It installs NemoClaw if needed, selects Hermes with `NEMOCLAW_AGENT=hermes`, then creates or verifies the sandbox named by `NEMOCLAW_SANDBOX_NAME`.

Notes:

- The installer may print `NemoHermes is ready` when onboarding succeeds.
- If the cell fails after an interrupted run, rerun it. If NemoClaw asks for a fresh session, set `NEMOCLAW_ONBOARD_EXTRA_ARGS=--fresh` in the config cell and run again.
- Keep this notebook open while the first build runs; pulling and building the Hermes image is the slow part.

In [ ]:
%%bash
set -euo pipefail
source /tmp/nemoclaw-hermes-launchable.env

load_nemoclaw_path() {
  if [ -s "$HOME/.nvm/nvm.sh" ]; then
    # shellcheck disable=SC1091
    . "$HOME/.nvm/nvm.sh"
  fi
  export PATH="$HOME/.local/bin:$HOME/.npm-global/bin:$PATH"
}

load_nemoclaw_path

if ! command -v nemohermes >/dev/null 2>&1; then
  echo "Installing NemoClaw and starting Hermes onboarding..."
  curl -fsSL https://www.nvidia.com/nemoclaw.sh | bash -s -- --yes-i-accept-third-party-software
  load_nemoclaw_path
fi

if ! command -v nemohermes >/dev/null 2>&1; then
  echo "nemohermes is not on PATH yet. Sourcing shell profile and retrying..."
  [ -f "$HOME/.bashrc" ] && . "$HOME/.bashrc" >/dev/null 2>&1 || true
  load_nemoclaw_path
fi

command -v nemohermes
nemohermes --version || true

if nemohermes "$NEMOCLAW_SANDBOX_NAME" status >/tmp/nemoclaw-hermes-status.txt 2>&1; then
  cat /tmp/nemoclaw-hermes-status.txt
  echo "Sandbox $NEMOCLAW_SANDBOX_NAME already exists. Skipping onboarding."
else
  echo "Creating Hermes sandbox $NEMOCLAW_SANDBOX_NAME..."
  # Optional recovery hook for interrupted sessions, for example:
  #   export NEMOCLAW_ONBOARD_EXTRA_ARGS=--fresh
  # in the first cell before rerunning this one.
  # shellcheck disable=SC2086
  nemohermes onboard \
    --non-interactive \
    --yes-i-accept-third-party-software \
    --yes \
    --name "$NEMOCLAW_SANDBOX_NAME" \
    ${NEMOCLAW_ONBOARD_EXTRA_ARGS:-}
fi

## Verify Hermes

Hermes exposes an OpenAI-compatible API on port `8642` and a health endpoint at `/health`. NemoClaw usually starts the forward automatically; this cell starts it again if needed, then checks status and health.

In [ ]:
%%bash
set -euo pipefail
source /tmp/nemoclaw-hermes-launchable.env

if [ -s "$HOME/.nvm/nvm.sh" ]; then . "$HOME/.nvm/nvm.sh"; fi
export PATH="$HOME/.local/bin:$HOME/.npm-global/bin:$PATH"

nemohermes "$NEMOCLAW_SANDBOX_NAME" status
openshell forward start --background 8642 "$NEMOCLAW_SANDBOX_NAME" >/dev/null 2>&1 || true

for i in $(seq 1 60); do
  if curl -sf http://127.0.0.1:8642/health; then
    echo
    echo "Hermes health endpoint is reachable at http://127.0.0.1:8642/health"
    echo "OpenAI-compatible base URL: http://127.0.0.1:8642/v1"
    exit 0
  fi
  sleep 2
done

echo "Hermes health endpoint was not reachable within 120 seconds." >&2
echo "Recent logs:" >&2
nemohermes "$NEMOCLAW_SANDBOX_NAME" logs --tail 80 || true
exit 1

## Launch the Hermes TUI

The Hermes TUI is interactive, so launch it from a terminal, not from a notebook cell.

In JupyterLab on Brev, open **File -> New -> Terminal** and run:

```bash
source /tmp/nemoclaw-hermes-launchable.env
nemohermes "$NEMOCLAW_SANDBOX_NAME" connect
```

Inside the sandbox shell, run:

```bash
hermes
```

That opens the Hermes terminal UI. Use the same terminal path for agent-owned files under `/sandbox/.hermes`, logs, skills, and workspace state.

In [ ]:
%%bash
set -euo pipefail
source /tmp/nemoclaw-hermes-launchable.env

cat <<EOF
Open a JupyterLab terminal on this Brev instance and run:

  source /tmp/nemoclaw-hermes-launchable.env
  nemohermes "$NEMOCLAW_SANDBOX_NAME" connect

Then, inside the sandbox:

  hermes

Useful host-side commands:

  nemohermes "$NEMOCLAW_SANDBOX_NAME" status
  nemohermes "$NEMOCLAW_SANDBOX_NAME" logs --follow
  openshell sandbox exec -n "$NEMOCLAW_SANDBOX_NAME" -- hermes --version
EOF

## OpenShell Term and Remote Access

`openshell term` is the OpenShell host TUI. Use it to watch sandbox activity, inspect network egress decisions, and approve requests when you run with policies that require approval.

From a JupyterLab terminal on the Brev instance:

```bash
openshell term
```

The terminal can be used to approve outbound network requests or view the currently active policies.

## Cleanup

Brev bills while the instance is running. Destroy the Hermes sandbox when you no longer need it, then stop or delete the Brev instance from the Brev console or CLI.

The cell below is guarded. Change `DESTROY_HERMES_SANDBOX` to `1` only when you really want to remove the sandbox.